In [3]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

In [4]:
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=1000, wiki_client=None)
# To run a wrapper
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [5]:
# Create the wrapper with a custom user agent in the client kwargs
api_wrapper = WikipediaAPIWrapper(
    top_k_results=3,
    doc_content_chars_max=4000,
    wiki_client=None
)

# Explicitly patch the underlying wikipedia package's headers if needed
import wikipedia
wikipedia.set_user_agent("MyDeepLearningProject/1.0 (your_email@example.com)")

# Initialize the tool
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

# Run it again
wiki.run("Albert Einstein")

'Page: Albert Einstein\nSummary: Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the known theory of relativity. Einstein also made important contributions to quantum theory. His mass–energy equivalence formula E = mc2, which arises from special relativity, has been called "the world\'s most famous equation". He received the 1921 Nobel Prize in Physics for "his services to theoretical physics, and especially for his discovery of the law of the photoelectric effect".\nBorn as a subject to the Kingdom of Württemberg, part of the German Empire, Einstein moved to Switzerland in 1895, forsaking his citizenship the following year. In 1897, at the age of seventeen, he enrolled in the mathematics and physics teaching diploma program at the Swiss federal polytechnic school in Zurich, graduating in 1900. He acquired Swiss citizenship a year later, which he kept for the rest of his life, and afterwards secured a permanent position 

In [6]:
import arxiv

custom_arxiv_client = arxiv.Client(
        page_size=3,          # Fetch only what you need, NOT the default 100
        delay_seconds=3.0,    # arXiv guidelines strictly require a 3-second gap
        num_retries=5,
    )

In [7]:
api_arxiv_wrapper = ArxivAPIWrapper(
    top_k_results=3, 
    doc_content_chars_max=4000,
    arxiv_client=custom_arxiv_client,
)
import time
# Create the tool
arxiv_tool = ArxivQueryRun(api_wrapper=api_arxiv_wrapper)

# Test it out! You can search by topics, keywords, or specific authors
result = arxiv_tool.run("Attention Is All You Need")
time.sleep(3)
print(result)

Published: 2021-05-06
Title: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet
Authors: Luke Melas-Kyriazi
Summary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifically, we replace the attention layer in a vision transformer with a feed-forward layer applied over the patch dimension. The resulting architecture is simply a series of feed-forward layers applied over the patch and feature dimensions in an alternating fashion. In experiments on ImageNet, this architecture performs surprisingly well: a ViT/DeiT-base-sized model obtains 74.9\% top-1 accuracy, compared to 77.9\% and 79.9\% for ViT and DeiT respectively. These results indicate that aspects of vis

In [8]:
print(arxiv_tool.name)

arxiv


In [9]:
tools = [wiki, arxiv_tool]

In [10]:
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [11]:
from langchain_core.tools import create_retriever_tool

In [12]:
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv(), override=True)

True

In [13]:
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vector_store = FAISS.from_documents(documents, OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY")))
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000025FAFF14650>, search_kwargs={'k': 2})

In [14]:
retriver_tool = create_retriever_tool(
    retriever,
    name="langsmith-search",
    description="Search any information about langsmith"
)

In [15]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\user\\Documents\\DeepLearning-Indepth\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000))]

In [16]:
tools.append(retriver_tool)
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\user\\Documents\\DeepLearning-Indepth\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000)),
 StructuredTool(name='langsmith-search', description='Search any information about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000025FAEEA84A0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000025FAF3D37E0>)]

In [17]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\user\\Documents\\DeepLearning-Indepth\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000)),
 StructuredTool(name='langsmith-search', description='Search any information about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000025FAEEA84A0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000025FAF3D37E0>)]

In [18]:
## Llm and agents

llm = ChatOpenAI(api_key=os.getenv("OPENAI_API_KEY"))
from langchain.agents import create_agent
from langchain_classic import hub

prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.pretty_print()

================================ System Message ================================

You are a helpful assistant

============================= Messages Placeholder =============================

{chat_history}

================================ Human Message =================================

{input}

============================= Messages Placeholder =============================

{agent_scratchpad}


In [19]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [20]:
## Using agents
from langchain_classic.agents import (AgentExecutor, 
    create_openai_functions_agent, create_openai_tools_agent)

agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [21]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
result = agent_executor.invoke({"input": "What is attention mechanism in AI?"})
print(result)



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'attention mechanism in AI'}`


Page: Attention Is All You Need
Summary: "Attention Is All You Need" is a 2017 research paper in machine learning authored by eight scientists and engineers working at Google. The paper introduced a new deep learning architecture known as the transformer, based on the attention mechanism proposed in 2014 by Bahdanau et al. The transformer approach it describes has become the main architecture of a wide variety of artificial intelligence systems, including large language models. At the time, the focus of the research was on improving Seq2seq techniques for machine translation, but the authors go further in the paper, foreseeing the technique's potential for other tasks like question answering and what is now known as multimodal generative AI.
Some early examples that the team tried their Transformer architecture on included English-to-German translation, generating Wikipedia ar

In [22]:
print(result["output"])

In AI, the attention mechanism is a method that determines the importance of each component in a sequence relative to others in that sequence. It assigns "soft" weights to each element in a sequence, representing their importance. This mechanism is widely used in natural language processing to encode vectors across sequences and plays a crucial role in tasks like machine translation and question answering.

The concept of attention in AI was popularized by the research paper "Attention Is All You Need," which introduced the transformer architecture. This architecture, based on the attention mechanism, has become a fundamental component in various artificial intelligence systems, especially in large language models. The transformer architecture eliminates the need for slower sequential RNNs and relies on a faster parallel attention scheme.

The attention mechanism allows tokens in a sequence to have equal access to any part of the sequence directly, rather than favoring recent informati

In [24]:
result = agent_executor.invoke({"input":"Find the paper: Attention is all you need"})
print(result['output'])



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': 'Attention is all you need'}`


Published: 2021-05-06
Title: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet
Authors: Luke Melas-Kyriazi
Summary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifically, we replace the attention layer in a vision transformer with a feed-forward layer applied over the patch dimension. The resulting architecture is simply a series of feed-forward layers applied over the patch and feature dimensions in an alternating fashion. In experiments on ImageNet, this architecture performs surprisingly well: a ViT/DeiT-base-sized model obtains 74.9\% top-1 accura

In [25]:
agent_executor.invoke({"input": "Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'Langsmith'}`


pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack.

LangSmith docs - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresLLM GatewayPrivate betaOverviewSpend policiesPII and secrets redactionAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizat

{'input': 'Tell me about Langsmith',
 'output': "LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM (Large Language Model) applications. It allows users to trace requests, evaluate outputs, test prompts, and manage deployments all in one place with their agent stack. If you'd like more detailed information or have specific questions about LangSmith, feel free to let me know!"}